# Forget-MI LoKU — Hyperparameter Sweep (Giảm Df_AUC ở 6%/10%)

> 🎯 **Mục đích**: Sweep 7 configs để tìm sweet-spot giảm Df_AUC ở **6% và 10%** mà không hi sinh quá nhiều Test AUC.
> 3% đã đạt kết quả tốt (run.ipynb) → KHÔNG chạy lại ở đây.

## Lý do sweep

Multi-seed run.ipynb cho thấy:
- 6%: Df_AUC=0.729 vs paper 0.654 → +0.075 ❌
- 10%: Df_AUC=0.781 vs paper 0.656 → +0.125 ❌

Df_AUC cao = forget chưa đủ mạnh. Cần tăng forget signal mà không phá utility.

## 7 Configs sweep

| ID | IHL | img_scale | extras | Test hypothesis |
|---|---|---|---|---|
| baseline | 0.75 | 0.3 | — | (đã có từ run.ipynb, dùng làm điểm tham chiếu) |
| **A_ihl100** | **1.0** | 0.3 | — | IHL bump alone |
| **B_img050** | 0.75 | **0.5** | — | Image-FILA bump alone |
| **C_combo_moderate** | **1.0** | **0.5** | — | Combined moderate (sweet candidate) |
| **D_combo_aggressive** | **1.25** | **0.5** | — | IHL aggressive on top of img bump |
| **E_extreme** | **1.5** | **0.7** | — | Most aggressive (risk over-forget) |
| **F_more_epochs** | 0.75 | 0.3 | epochs=12 | "Just train longer" |
| **G_less_retain** | 0.75 | 0.3 | kappa_cls_retain=1.0 | "Less retain anchor" |

## Workflow

```
Cell 1-3.5  : setup (skip Cell 2-3 nếu data đã có sẵn)
Cell 3.6    : load helpers (PAPER_REF, GOLD_RETRAINED, ...)
Cell 4      : SWEEP_CONFIGS + helpers run_sweep / aggregate_sweep
Cell 5      : 🚀 Run sweep 6% (7 configs × 1 seed) — ~1.5 giờ
Cell 6      : 🚀 Run sweep 10% (7 configs × 1 seed) — ~1.5 giờ
Cell 7      : Bảng tổng hợp + recommend best config / forget%
Cell 8      : (tuỳ chọn) Multi-seed cho config WINNING
Cell 9      : Push GitHub
```

## Lưu ý

- Notebook này **KHÔNG ghi đè run.ipynb** — chạy độc lập, tracker vẫn append vào INDEX cũ.
- Tracker tạo `exp_NNN_sweep_<pct>per_<config_id>.md` cho mỗi run.
- CSV chung `unlearning_output/results_summary.csv` được append (cả baseline + sweep).
- Output đặc thù: `experiments/summary_sweep_<pct>per.md` (so các config) + `experiments/bang_sweep_recommendations.md`.

In [ ]:
# ====================================
# CELL 1: Kết nối Drive & Pull Code (giữ data)
# ====================================
from google.colab import drive
import os

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)
else:
    print("✅ Google Drive đã được kết nối!")

%cd /content
REPO = "Forget-MI-LoKU"
REPO_URL = "https://github.com/nhnhu146/Forget-MI-LoKU.git"

if not os.path.exists(REPO):
    !git clone {REPO_URL}
else:
    %cd {REPO}
    !git fetch origin
    !git reset --hard origin/master 2>&1 | tail -3
    %cd /content

%cd {REPO}
!git log --oneline -1

# Check ALL critical packages (peft + pydicom — fix bug 2026-06-18)
# Colab session có thể giữ peft từ session trước nhưng mất pydicom
import importlib.util
required = ["peft", "pydicom", "transformers", "accelerate"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    print(f"📦 Thiếu: {missing}. Đang cài...")
    !pip install -q pydicom scikit-image wandb pyyaml pandas
    !pip install -q "transformers==4.38.0" "peft==0.10.0" "accelerate==0.27.0"
else:
    print("✅ Đã có đủ peft + pydicom + transformers + accelerate.")

import torch
if torch.cuda.is_available():
    print(f"\n🟢 GPU OK: {torch.cuda.get_device_name(0)}")
else:
    print("\n🔴 KHÔNG CÓ GPU — sweep sẽ treo! Bật GPU rồi rerun.")

print("\n✅ Sẵn sàng. Skip Cell 2-3 nếu data đã extract sẵn.")

In [ ]:
# ====================================
# CELL 2: Extract data (CHỈ CHẠY LẦN ĐẦU)
# ====================================
!python setup_data.py

In [ ]:
# ====================================
# CELL 3: Tiền xử lý + symlink output (CHẠY MỖI SESSION COLAB!)
# ====================================
# ⚠️ QUAN TRỌNG: symlink `unlearning_output/` → Drive bị MẤT khi Colab restart.
# Chạy lại Cell 3 mỗi session để tạo lại symlink → CSV trên Drive được "thấy".
# Đây là cơ chế CHECKPOINT — CSV trên Drive cho phép resume sweep nếu Colab cut.

import os, shutil

# 1. Tạo all_data.tsv (idempotent, skip nếu đã có)
!python make_tsv.py

# 2. Tạo + symlink unlearning_output về Drive (LÀM MỖI SESSION)
DRIVE_RESULTS = "/content/drive/MyDrive/Forget-MI-Project/unlearning_output"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

if os.path.exists("unlearning_output"):
    if os.path.islink("unlearning_output"):
        os.unlink("unlearning_output")
    else:
        shutil.rmtree("unlearning_output")
!ln -s "{DRIVE_RESULTS}" ./unlearning_output

# 3. Tạo + symlink experiments_backup về Drive (backup tracker MDs)
DRIVE_EXP_BACKUP = "/content/drive/MyDrive/Forget-MI-Project/experiments_backup"
os.makedirs(DRIVE_EXP_BACKUP, exist_ok=True)

# Restore tracker MDs từ Drive backup nếu có (cho session sau restart)
os.makedirs("experiments", exist_ok=True)
restored = 0
for f in os.listdir(DRIVE_EXP_BACKUP):
    if f.endswith('.md'):
        src = os.path.join(DRIVE_EXP_BACKUP, f)
        dst = os.path.join("experiments", f)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            restored += 1
if restored > 0:
    print(f"\n♻️  Restored {restored} tracker MD files từ Drive backup")

# 4. Check CSV state — show resume info
# Backward-compat: CSV cũ KHÔNG có 'id' (fix 2026-06-18). Hiển thị thông tin
# dựa trên các columns có sẵn (forget_pct + hyperparams).
csv_path = "unlearning_output/results_summary.csv"
if os.path.exists(csv_path):
    import pandas as pd
    df = pd.read_csv(csv_path)
    print(f"\n📊 CSV tại {csv_path}")
    print(f"   Total rows: {len(df)}")
    print(f"   Columns: {list(df.columns)[:10]}{'...' if len(df.columns)>10 else ''}")

    # Show breakdown theo forget_pct
    if 'forget_pct' in df.columns:
        for pct in [3, 6, 10]:
            pct_rows = df[df['forget_pct'].astype(str).str.contains(f"_{pct}per")]
            if len(pct_rows) == 0:
                continue
            # Phân loại: baseline (ihl≈0.75, img≈0.3) vs sweep (others)
            if 'ihl' in df.columns and 'img_subtract' in df.columns:
                base_mask = ((pct_rows['ihl'].astype(float).round(2) == 0.75)
                             & (pct_rows['img_subtract'].astype(float).round(2) == 0.30))
                n_base = base_mask.sum()
                n_sweep = len(pct_rows) - n_base
                breakdown = f"baseline={n_base}, sweep_variants={n_sweep}"
            else:
                breakdown = f"total={len(pct_rows)}"

            # Nếu có 'id' column thì list distinct ids
            if 'id' in df.columns:
                ids = pct_rows['id'].dropna().unique().tolist()
                # Filter empty/nan strings
                ids = [i for i in ids if i and str(i) != 'nan']
                if ids:
                    breakdown += f" | ids: {ids}"
            print(f"     • {pct}%: {breakdown}")

    if 'id' not in df.columns:
        print(f"\n   ⚠️  CSV cũ KHÔNG có column 'id' (trước fix 2026-06-18)")
        print(f"      → Sweep helper sẽ dùng FALLBACK hyperparam match để skip duplicate.")
        print(f"      → Sau sweep mới, CSV sẽ có cả rows mới (có 'id') và rows cũ (không có 'id').")
else:
    print(f"\n📊 Chưa có CSV — sweep sẽ chạy từ đầu")

print(f"\n✅ Symlink OK: ./unlearning_output → {DRIVE_RESULTS}")
print(f"✅ Backup OK: ./experiments backups → {DRIVE_EXP_BACKUP}")
print(f"💡 Colab cut giữa sweep → rerun Cell 1 → Cell 3 → Cell 5/6 → auto-skip configs đã xong.")

In [ ]:
# ====================================
# CELL 3.5: Verify config — confirm honest mode
# ====================================
!grep -E "^\s*(distill_teacher|early_stop_metric|distill_forget_weight|eta_re_anchor):" -A 1 config.yaml | grep "value:" | xargs -I{} echo "   → {}"
print("   ✅ distill_teacher=og, early_stop_metric=val, distill_forget_weight=0, eta_re_anchor=0")
!git log --oneline -1

In [ ]:
# ====================================
# CELL 3.6: Base helpers (PAPER_REF, GOLD_RETRAINED, METRIC_DEFS)
# ====================================
# Subset từ run.ipynb — chỉ thứ cần cho sweep aggregator.

import os, numpy as np, pandas as pd
from datetime import datetime

CSV_PATH = "unlearning_output/results_summary.csv"

PAPER_REF = {
    3:  {"MIA_paper": 0.571, "Df_AUC": 0.735, "Df_F1": 0.393, "Dt_AUC": 0.625, "Dt_F1": 0.250, "Time_h": 5.0},
    6:  {"MIA_paper": 0.615, "Df_AUC": 0.654, "Df_F1": 0.328, "Dt_AUC": 0.599, "Dt_F1": 0.270, "Time_h": 5.0},
    10: {"MIA_paper": 0.810, "Df_AUC": 0.656, "Df_F1": 0.313, "Dt_AUC": 0.565, "Dt_F1": 0.252, "Time_h": 5.0},
}

GOLD_RETRAINED = {
    3:  "./model_retrained_3per/",
    6:  None,
    10: None,
}

# Baseline config (cho cột reference trong sweep table + fallback id detection)
BASELINE_PARAMS = {
    'ihl': 0.75,
    'img_scale': 0.3,
    'epochs': 8,
    'kappa': 2.0,
}


def _check_gold(forget_pct):
    p = GOLD_RETRAINED.get(forget_pct)
    if p is None: return None, False
    return p, os.path.isdir(p)


def get_baseline_row(forget_pct):
    """Đọc baseline rows từ CSV (đã chạy ở run.ipynb).

    Backward-compat: CSV cũ KHÔNG có column 'id' (fix 2026-06-18).
    → Fallback: nhận diện baseline bằng hyperparameter match (ihl=0.75, img=0.3).
    """
    if not os.path.exists(CSV_PATH):
        return None
    df = pd.read_csv(CSV_PATH)
    if 'forget_pct' not in df.columns:
        return None

    pct_mask = df['forget_pct'].astype(str).str.contains(f"_{forget_pct}per")

    # Preferred: dùng 'id' column nếu có (CSV mới)
    if 'id' in df.columns:
        id_mask = pct_mask & (df['id'].astype(str) == f'loku_{forget_pct}per')
        sub = df[id_mask]
        if not sub.empty:
            return sub

    # Fallback: hyperparam match (CSV cũ không có 'id')
    if 'ihl' in df.columns and 'img_subtract' in df.columns:
        # Baseline = ihl=0.75 & img_subtract=0.3 (default config từ run.ipynb)
        hp_mask = (pct_mask
                   & (df['ihl'].astype(float).round(2) == BASELINE_PARAMS['ihl'])
                   & (df['img_subtract'].astype(float).round(2) == BASELINE_PARAMS['img_scale']))
        sub = df[hp_mask]
        if not sub.empty:
            return sub

    # Last resort: tất cả rows của forget% này (CSV chỉ có baseline runs)
    return df[pct_mask] if pct_mask.any() else None


print("✅ Base helpers loaded (with backward-compat for CSV without 'id' column)")
print(f"   CSV: {CSV_PATH}")
print(f"   Gold retrained: 3% ✅ | 6% ❌ | 10% ❌")

In [ ]:
# ====================================
# CELL 4: SWEEP_CONFIGS + sweep helpers (RESUME-CAPABLE)
# ====================================
# Mỗi config có 2 lớp checkpoint:
#   1. CSV row trên Drive (✅ luôn auto-save, làm CSV-based skip work)
#   2. Tracker MD trên Drive backup (♻️ auto-copy sau mỗi run)
#
# BACKWARD COMPAT (fix 2026-06-18):
#   - CSV rows TRƯỚC fix không có column 'id' → _sweep_seed_done fallback
#     dùng hyperparam match (ihl + img_subtract + epochs + kappa)
#   - CSV rows SAU fix có 'id' → preferred id match

import os, shutil, numpy as np, pandas as pd
from datetime import datetime
from pathlib import Path

DRIVE_EXP_BACKUP = "/content/drive/MyDrive/Forget-MI-Project/experiments_backup"

SWEEP_CONFIGS = {
    'A_ihl100': {
        'ihl': 1.0, 'img_scale': 0.3, 'epochs': 8, 'kappa': 2.0,
        'note': 'IHL bump alone (test thuan IHL effect)',
    },
    'B_img050': {
        'ihl': 0.75, 'img_scale': 0.5, 'epochs': 8, 'kappa': 2.0,
        'note': 'Image-FILA bump alone (MIA chi doc img_logits)',
    },
    'C_combo_moderate': {
        'ihl': 1.0, 'img_scale': 0.5, 'epochs': 8, 'kappa': 2.0,
        'note': 'Combined moderate - sweet-spot candidate',
    },
    'D_combo_aggressive': {
        'ihl': 1.25, 'img_scale': 0.5, 'epochs': 8, 'kappa': 2.0,
        'note': 'IHL aggressive + img bump',
    },
    'E_extreme': {
        'ihl': 1.5, 'img_scale': 0.7, 'epochs': 8, 'kappa': 2.0,
        'note': 'Most aggressive - risk over-forget (Test AUC co the giam)',
    },
    'F_more_epochs': {
        'ihl': 0.75, 'img_scale': 0.3, 'epochs': 12, 'kappa': 2.0,
        'note': 'Baseline + epochs 8->12 (test co hoi tu them khong)',
    },
    'G_less_retain': {
        'ihl': 0.75, 'img_scale': 0.3, 'epochs': 8, 'kappa': 1.0,
        'note': 'Baseline + kappa_cls_retain 2.0->1.0 (less retain anchor)',
    },
}

SWEEP_METRIC_DEFS = [
    ('MIA_paper',           'MIA',          '↓'),
    ('Df_AUC',              'Df_AUC',       '↓'),
    ('Df_F1',               'Df_F1',        '↓'),
    ('Dt_AUC',              'Dt_AUC',       '↑'),
    ('Dt_F1',               'Dt_F1',        '↑'),
    ('forget_ce',           'forget_ce',    '·'),
    ('test_ce',             'test_ce',      '·'),
    ('unlearn_time_hours',  'Time(h)',      '↓'),
]


def _backup_experiments_to_drive():
    """Copy MD files mới sang Drive backup → checkpoint MDs."""
    if not os.path.isdir(DRIVE_EXP_BACKUP):
        os.makedirs(DRIVE_EXP_BACKUP, exist_ok=True)
    if not os.path.isdir("experiments"):
        return 0
    backed_up = 0
    for f in os.listdir("experiments"):
        if not f.endswith('.md'):
            continue
        src = os.path.join("experiments", f)
        dst = os.path.join(DRIVE_EXP_BACKUP, f)
        if not os.path.exists(dst) or os.path.getmtime(src) > os.path.getmtime(dst):
            shutil.copy2(src, dst)
            backed_up += 1
    return backed_up


def _hp_match_sweep(df, forget_pct, config, seed):
    """Match CSV rows bằng (forget_pct, seed, ihl, img_subtract[, epochs, kappa_cls_retain]).
    Dùng làm fallback khi CSV cũ không có 'id' column."""
    if 'forget_pct' not in df.columns or 'seed' not in df.columns:
        return df.iloc[0:0]
    mask = (df['forget_pct'].astype(str).str.contains(f"_{forget_pct}per")
            & (df['seed'] == seed))
    if 'ihl' in df.columns:
        mask = mask & (df['ihl'].astype(float).round(2) == round(config['ihl'], 2))
    if 'img_subtract' in df.columns:
        mask = mask & (df['img_subtract'].astype(float).round(2) == round(config['img_scale'], 2))
    # epochs/kappa chỉ check nếu CSV mới có (fix 2026-06-18 thêm vào)
    if 'epochs' in df.columns:
        mask = mask & (df['epochs'].astype(int) == int(config['epochs']))
    if 'kappa_cls_retain' in df.columns:
        mask = mask & (df['kappa_cls_retain'].astype(float).round(2) == round(config['kappa'], 2))
    return df[mask]


def _sweep_seed_done(forget_pct, config_id, seed):
    """Check (forget%, config, seed) đã có trong CSV chưa (checkpoint check).

    BACKWARD COMPAT: thử 'id' column trước (CSV mới), fallback hyperparam (CSV cũ).
    """
    if not os.path.exists(CSV_PATH):
        return False
    try:
        df = pd.read_csv(CSV_PATH)
    except Exception:
        return False
    if 'seed' not in df.columns or 'forget_pct' not in df.columns:
        return False

    # Preferred: 'id' column match (CSV mới sau fix 2026-06-18)
    if 'id' in df.columns:
        expected_id = f"sweep_{forget_pct}per_{config_id}"
        mask = (df['id'].astype(str) == expected_id) & (df['seed'] == seed)
        if mask.any():
            return True
        # Nếu id column tồn tại nhưng không match, có thể row được ghi TRƯỚC fix với id=NaN
        # → tiếp tục thử hyperparam fallback

    # Fallback: hyperparam match (CSV cũ KHÔNG có 'id')
    config = SWEEP_CONFIGS.get(config_id)
    if not config:
        return False
    sub = _hp_match_sweep(df, forget_pct, config, seed)
    return not sub.empty


def run_sweep_config(forget_pct, config_id, config, seed=42):
    """Chạy 1 config × 1 forget% × 1 seed + backup MDs."""
    forget_csv = f"./data_splits/forget_set_{forget_pct}per.csv"
    retrained_path, has_gold = _check_gold(forget_pct)
    exp_name = f"sweep_{forget_pct}per_{config_id}_seed{seed}"

    ovr_parts = [
        f"forget_set_path={forget_csv}",
        f"id=sweep_{forget_pct}per_{config_id}",
        f"ihl_forget_weight={config['ihl']}",
        f"loku_image_subtract_scale={config['img_scale']}",
        f"unlearn_epochs={config['epochs']}",
        f"kappa_cls_retain={config['kappa']}",
    ]
    if has_gold:
        ovr_parts.append(f"retrained_model_path={retrained_path}")
    OVR = ",".join(ovr_parts)

    HYPOTHESIS = (f"Sweep {config_id} @ {forget_pct}% seed={seed}. "
                  f"{config['note']}. "
                  f"IHL={config['ihl']}, img_scale={config['img_scale']}, "
                  f"epochs={config['epochs']}, kappa={config['kappa']}.")

    cmd = (f'PYTHONPATH=. WANDB_MODE=disabled python training/forgetmi_loku.py '
           f'--config config.yaml --fresh --seed {seed} '
           f'--override "{OVR}" '
           f'--exp {exp_name} --hypothesis "{HYPOTHESIS}"')
    print(f"\n{'='*68}\n🚀 SWEEP {config_id} @ {forget_pct}% seed={seed}\n{'='*68}")
    print(f"   IHL={config['ihl']}, img_scale={config['img_scale']}, "
          f"epochs={config['epochs']}, kappa={config['kappa']}")
    print(f"   Note: {config['note']}\n")
    get_ipython().system(cmd)

    n = _backup_experiments_to_drive()
    if n > 0:
        print(f"♻️  Đã backup {n} MD file(s) → Drive ({DRIVE_EXP_BACKUP})")


def run_sweep(forget_pct, configs=None, seeds=(42,), force_redo=False):
    """Loop tất cả configs × seeds cho 1 forget%."""
    if configs is None:
        configs = SWEEP_CONFIGS

    done_count = sum(1 for cid in configs for s in seeds
                     if _sweep_seed_done(forget_pct, cid, s))
    total = len(configs) * len(seeds)

    print(f"\n{'#'*72}")
    print(f"# 🎯 SWEEP @ FORGET {forget_pct}% — {len(configs)} configs × {len(seeds)} seed(s)")
    print(f"# Status: {done_count}/{total} đã xong (skip), {total-done_count} cần chạy")
    print(f"# Dự kiến còn lại: ~{12 * (total-done_count)} phút")
    print(f"# Configs: {list(configs.keys())}")
    print(f"{'#'*72}\n")

    for config_id, config in configs.items():
        for seed in seeds:
            if not force_redo and _sweep_seed_done(forget_pct, config_id, seed):
                print(f"⏭️  {config_id} seed {seed} đã có trong CSV — bỏ qua (checkpoint hit)")
                continue
            run_sweep_config(forget_pct, config_id, config, seed)

    _backup_experiments_to_drive()
    aggregate_sweep(forget_pct, configs, seeds)


def aggregate_sweep(forget_pct, configs, seeds=(42,)):
    """Print bảng so các configs + lưu MD + recommend best.

    Backward-compat: dùng 'id' nếu có, fallback hyperparam nếu CSV cũ.
    """
    if not os.path.exists(CSV_PATH):
        print(f"❌ {CSV_PATH} không tồn tại")
        return

    df_all = pd.read_csv(CSV_PATH)
    paper = PAPER_REF.get(forget_pct, {})
    has_id_col = 'id' in df_all.columns

    rows = []

    # Baseline row (từ run.ipynb)
    base = get_baseline_row(forget_pct)
    baseline_means = {}
    if base is not None and not base.empty:
        if 'timestamp' in base.columns:
            base = base.sort_values('timestamp').drop_duplicates(subset=['seed'], keep='last')
        for csv_k, _, _ in SWEEP_METRIC_DEFS:
            if csv_k in base.columns:
                vals = base[csv_k].dropna().values
                if len(vals) > 0:
                    baseline_means[csv_k] = float(np.mean(vals))
        rows.append({
            'config_id': f'baseline (n={len(base)})',
            'IHL': BASELINE_PARAMS['ihl'],
            'img': BASELINE_PARAMS['img_scale'],
            'ep': BASELINE_PARAMS['epochs'],
            'kappa': BASELINE_PARAMS['kappa'],
            **baseline_means,
        })

    # Paper row
    rows.append({
        'config_id': 'paper',
        'IHL': '—', 'img': '—', 'ep': '—', 'kappa': '—',
        'MIA_paper': paper.get('MIA_paper'),
        'Df_AUC': paper.get('Df_AUC'),
        'Df_F1': paper.get('Df_F1'),
        'Dt_AUC': paper.get('Dt_AUC'),
        'Dt_F1': paper.get('Dt_F1'),
        'unlearn_time_hours': paper.get('Time_h'),
    })

    # Sweep rows — preferred id match, fallback hyperparam
    for config_id, config in configs.items():
        sub = pd.DataFrame()
        if has_id_col:
            sweep_id = f"sweep_{forget_pct}per_{config_id}"
            id_mask = (df_all['id'].astype(str) == sweep_id) & df_all['seed'].isin(seeds)
            sub = df_all[id_mask]
        if sub.empty:
            # Fallback hyperparam
            sub_all_seeds = pd.concat([_hp_match_sweep(df_all, forget_pct, config, s)
                                       for s in seeds], ignore_index=False)
            sub = sub_all_seeds

        if sub.empty:
            rows.append({'config_id': config_id, 'IHL': config['ihl'], 'img': config['img_scale'],
                         'ep': config['epochs'], 'kappa': config['kappa']})
            continue
        if 'timestamp' in sub.columns:
            sub = sub.sort_values('timestamp').drop_duplicates(subset=['seed'], keep='last')
        row = {'config_id': config_id, 'IHL': config['ihl'], 'img': config['img_scale'],
               'ep': config['epochs'], 'kappa': config['kappa']}
        for csv_k, _, _ in SWEEP_METRIC_DEFS:
            if csv_k in sub.columns:
                vals = sub[csv_k].dropna().values
                if len(vals) > 0:
                    row[csv_k] = float(np.mean(vals))
        if 'MIA_paper' in row and 'Dt_AUC' in row:
            row['PUS'] = (1 - row['MIA_paper']) * row['Dt_AUC']
        rows.append(row)

    # ----- Print table -----
    print(f"\n{'='*120}")
    print(f"📊 SWEEP RESULTS — FORGET {forget_pct}% (seeds={list(seeds)})")
    print(f"{'='*120}")

    hdr = (f"{'Config':<22}{'IHL':>5}{'img':>6}{'ep':>4}{'kp':>5}"
           f"{'MIA↓':>8}{'Df_AUC↓':>10}{'Df_F1↓':>9}{'Dt_AUC↑':>10}{'Dt_F1↑':>9}{'f_ce':>7}{'t_ce':>7}{'Time':>7}{'PUS↑':>8}")
    print(hdr); print("-" * len(hdr))

    md_rows = [
        "| Config | IHL | img | ep | κ | MIA↓ | Df_AUC↓ | Df_F1↓ | Dt_AUC↑ | Dt_F1↑ | f_ce | t_ce | Time | PUS↑ |",
        "|---|---|---|---|---|---|---|---|---|---|---|---|---|---|",
    ]

    def _f(v, decimals=3):
        if v is None or (isinstance(v, float) and v != v): return '—'
        if isinstance(v, str): return v
        return f"{v:.{decimals}f}"

    for r in rows:
        cid = r['config_id']
        cid_disp = cid
        if cid == 'paper': cid_disp = f"📜 {cid}"
        elif cid.startswith('baseline'): cid_disp = f"⚖️  {cid}"

        row_vals = [_f(r.get(k), d) for k, d in [
            ('IHL', 2), ('img', 2), ('ep', 0), ('kappa', 1),
            ('MIA_paper', 3), ('Df_AUC', 3), ('Df_F1', 3),
            ('Dt_AUC', 3), ('Dt_F1', 3),
            ('forget_ce', 2), ('test_ce', 2),
            ('unlearn_time_hours', 2), ('PUS', 3),
        ]]
        print(f"{cid_disp:<22}{row_vals[0]:>5}{row_vals[1]:>6}{row_vals[2]:>4}{row_vals[3]:>5}"
              f"{row_vals[4]:>8}{row_vals[5]:>10}{row_vals[6]:>9}{row_vals[7]:>10}{row_vals[8]:>9}"
              f"{row_vals[9]:>7}{row_vals[10]:>7}{row_vals[11]:>7}{row_vals[12]:>8}")
        md_rows.append(f"| {cid} | " + " | ".join(row_vals) + " |")

    # ----- Recommend best -----
    sweep_rows = [r for r in rows if r['config_id'] not in ('paper',) and not r['config_id'].startswith('baseline')]
    sweep_rows = [r for r in sweep_rows if r.get('PUS') is not None]
    if sweep_rows:
        sweep_rows.sort(key=lambda r: r.get('PUS', 0), reverse=True)
        best = sweep_rows[0]
        print(f"\n🏆 BEST by PUS (Privacy×Utility): {best['config_id']} → PUS={best['PUS']:.3f}")
        if 'MIA_paper' in baseline_means and 'Dt_AUC' in baseline_means:
            base_pus = (1 - baseline_means['MIA_paper']) * baseline_means['Dt_AUC']
            delta = best['PUS'] - base_pus
            sym = '✅' if delta > 0 else '❌'
            print(f"   vs baseline PUS={base_pus:.3f}: {sym} {delta:+.3f}")
        print(f"\n📋 Top 3 by PUS:")
        for i, r in enumerate(sweep_rows[:3], 1):
            print(f"   {i}. {r['config_id']:<22} PUS={r['PUS']:.3f}, "
                  f"MIA={_f(r.get('MIA_paper'))}, Df_AUC={_f(r.get('Df_AUC'))}, "
                  f"Dt_AUC={_f(r.get('Dt_AUC'))}")

    # ----- Save MD -----
    os.makedirs("experiments", exist_ok=True)
    out_md = f"experiments/summary_sweep_{forget_pct}per.md"
    body = [
        f"# Sweep Summary — FORGET {forget_pct}% (1 seed test)",
        "",
        f"_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}_",
        "",
        f"**Seed**: {list(seeds)}",
        f"**Configs tested**: {len(configs)}",
        "",
        f"**Paper Forget-MI ({forget_pct}%)**: MIA={paper.get('MIA_paper')}, "
        f"Df_AUC={paper.get('Df_AUC')}, Dt_AUC={paper.get('Dt_AUC')}",
        "",
        "## Comparison Table",
        "",
        *md_rows,
        "",
        "**PUS** = (1 − MIA) × Dt_AUC — higher = better privacy + utility balance",
    ]
    if sweep_rows:
        body += [
            "",
            "## Recommendation",
            "",
            f"🏆 **Best by PUS**: `{sweep_rows[0]['config_id']}` (PUS={sweep_rows[0]['PUS']:.3f})",
        ]
    with open(out_md, 'w', encoding='utf-8') as f:
        f.write('\n'.join(body))
    print(f"\n💾 Summary MD: {out_md}")
    _backup_experiments_to_drive()


print(f"✅ Sweep helpers loaded (with backward-compat for CSV without 'id').")
print(f"   {len(SWEEP_CONFIGS)} configs available:")
for k, v in SWEEP_CONFIGS.items():
    print(f"   • {k:<22} IHL={v['ihl']}, img={v['img_scale']}, ep={v['epochs']}, κ={v['kappa']}")
print(f"\n♻️  Auto-backup mỗi run → {DRIVE_EXP_BACKUP}")

In [ ]:
# ====================================
# CELL 5: 🚀 SWEEP @ FORGET 6% — 7 configs × 1 seed (~1.5 giờ)
# ====================================
# Mỗi config ~12 phút × 7 = ~84 phút.
# Nếu muốn giảm thời gian, comment bớt SWEEP_CONFIGS ở Cell 4.
#
# Helper tự skip configs đã chạy → safe nếu Colab disconnect.

run_sweep(forget_pct=6, seeds=(42,))

In [ ]:
# ====================================
# CELL 6: 🚀 SWEEP @ FORGET 10% — 7 configs × 1 seed (~1.5 giờ)
# ====================================
# Đây là case khó nhất (Df_AUC gap +0.125 so paper).
# Kỳ vọng: configs C/D/E (combined) sẽ giảm Df_AUC đáng kể.

run_sweep(forget_pct=10, seeds=(42,))

In [ ]:
# ====================================
# CELL 7: Cross-forget% recommendation — chọn config WINNING cho mỗi forget%
# ====================================
# Đọc CSV → cho mỗi forget% (6, 10), pick config có PUS cao nhất.
# Output: experiments/bang_sweep_recommendations.md
# Backward-compat: dùng 'id' nếu có, fallback hyperparam nếu CSV cũ.

import os, numpy as np, pandas as pd
from datetime import datetime

if not os.path.exists(CSV_PATH):
    print(f"❌ {CSV_PATH} không tồn tại")
else:
    df_all = pd.read_csv(CSV_PATH)
    has_id_col = 'id' in df_all.columns
    print(f"📋 Total CSV rows: {len(df_all)}, has 'id' column: {has_id_col}\n")

    md = [
        "# Sweep Recommendations — Pick Best Config per Forget%",
        "",
        f"_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}_",
        "",
        "Mục đích: chọn config có **PUS = (1−MIA) × Dt_AUC** cao nhất cho mỗi forget% ",
        "(balance privacy + utility), sau đó chạy multi-seed cho config này.",
        "",
    ]

    recommendations = {}
    for pct in [6, 10]:
        print(f"━━━ FORGET {pct}% ━━━")
        paper = PAPER_REF[pct]

        # Get baseline (with backward-compat fallback)
        base = get_baseline_row(pct)
        baseline_pus = None
        if base is not None and not base.empty:
            if 'MIA_paper' in base.columns and 'Dt_AUC' in base.columns:
                base_mia = base['MIA_paper'].dropna().mean()
                base_dt = base['Dt_AUC'].dropna().mean()
                if not pd.isna(base_mia) and not pd.isna(base_dt):
                    baseline_pus = (1 - base_mia) * base_dt
                    print(f"  Baseline (n={len(base)}): MIA={base_mia:.3f}, Dt_AUC={base_dt:.3f}, PUS={baseline_pus:.3f}")

        # Get sweep rows — preferred id, fallback hyperparam
        sweep_rows = []
        for config_id in SWEEP_CONFIGS:
            config = SWEEP_CONFIGS[config_id]
            sub = pd.DataFrame()
            if has_id_col:
                sweep_id = f"sweep_{pct}per_{config_id}"
                id_mask = df_all['id'].astype(str) == sweep_id
                sub = df_all[id_mask]
            if sub.empty:
                # Fallback hyperparam (cho tất cả seeds — Cell 7 không bind cụ thể seed)
                sub_pct = df_all[df_all['forget_pct'].astype(str).str.contains(f"_{pct}per")]
                if 'ihl' in sub_pct.columns and 'img_subtract' in sub_pct.columns:
                    mask = ((sub_pct['ihl'].astype(float).round(2) == round(config['ihl'], 2))
                            & (sub_pct['img_subtract'].astype(float).round(2) == round(config['img_scale'], 2)))
                    if 'epochs' in sub_pct.columns:
                        mask = mask & (sub_pct['epochs'].astype(int) == int(config['epochs']))
                    if 'kappa_cls_retain' in sub_pct.columns:
                        mask = mask & (sub_pct['kappa_cls_retain'].astype(float).round(2) == round(config['kappa'], 2))
                    sub = sub_pct[mask]

            if sub.empty:
                continue

            if 'timestamp' in sub.columns:
                sub = sub.sort_values('timestamp').drop_duplicates(subset=['seed'], keep='last')
            r = sub.iloc[-1]
            if pd.isna(r.get('MIA_paper')) or pd.isna(r.get('Dt_AUC')):
                continue
            pus = (1 - r['MIA_paper']) * r['Dt_AUC']
            sweep_rows.append({
                'config_id': config_id,
                'IHL': config['ihl'],
                'img': config['img_scale'],
                'epochs': config['epochs'],
                'MIA': float(r['MIA_paper']),
                'Df_AUC': float(r['Df_AUC']) if 'Df_AUC' in r and not pd.isna(r['Df_AUC']) else float('nan'),
                'Dt_AUC': float(r['Dt_AUC']),
                'PUS': float(pus),
            })

        if not sweep_rows:
            print(f"  ⚠️  Chưa có sweep nào @ {pct}% — chạy Cell 5/6 trước.\n")
            md.append(f"## {pct}%\n\n_Chưa có data_\n")
            continue

        sweep_rows.sort(key=lambda r: r['PUS'], reverse=True)
        best = sweep_rows[0]
        recommendations[pct] = best['config_id']

        print(f"  Sweep results sorted by PUS:")
        for i, r in enumerate(sweep_rows[:3], 1):
            print(f"    {i}. {r['config_id']:<22} PUS={r['PUS']:.3f}, MIA={r['MIA']:.3f}, "
                  f"Df_AUC={r['Df_AUC']:.3f}, Dt_AUC={r['Dt_AUC']:.3f}")

        delta_vs_base = f" (vs baseline {baseline_pus:.3f}: {best['PUS']-baseline_pus:+.3f})" if baseline_pus else ""
        print(f"  🏆 Best: {best['config_id']} → PUS={best['PUS']:.3f}{delta_vs_base}\n")

        md += [
            f"## {pct}% — Best: `{best['config_id']}`",
            "",
            f"**Recommendation**: PUS={best['PUS']:.3f}"
            + (f" (baseline={baseline_pus:.3f}, Δ={best['PUS']-baseline_pus:+.3f})" if baseline_pus else ""),
            "",
            f"Config: IHL={best['IHL']}, img_scale={best['img']}, epochs={best['epochs']}",
            "",
            "### Top 3 (sorted by PUS):",
            "",
            "| Rank | Config | IHL | img | MIA | Df_AUC | Dt_AUC | PUS |",
            "|---|---|---|---|---|---|---|---|",
        ]
        for i, r in enumerate(sweep_rows[:3], 1):
            md.append(f"| {i} | {r['config_id']} | {r['IHL']} | {r['img']} | {r['MIA']:.3f} "
                      f"| {r['Df_AUC']:.3f} | {r['Dt_AUC']:.3f} | **{r['PUS']:.3f}** |")
        md.append("")

    md += [
        "",
        "## Next Step: Multi-seed best configs",
        "",
        "```python",
    ]
    for pct, cid in recommendations.items():
        md.append(f"run_sweep(forget_pct={pct}, configs={{'{cid}': SWEEP_CONFIGS['{cid}']}}, seeds=(42, 123, 7))")
    md.append("```")

    out_md = "experiments/bang_sweep_recommendations.md"
    with open(out_md, 'w', encoding='utf-8') as f:
        f.write('\n'.join(md))
    print(f"💾 Recommendations saved: {out_md}")
    if recommendations:
        print(f"\n🎯 Next step: chạy Cell 8 với configs winning:")
        for pct, cid in recommendations.items():
            print(f"   • {pct}%: {cid}")

In [ ]:
# ====================================
# CELL 8: (Tuỳ chọn) Multi-seed cho config WINNING
# ====================================
# Sau khi Cell 7 đã pick best config, chạy cell này với 3 seeds để báo cáo mean±std.
# SỬA các tên config dưới đây theo recommendation Cell 7 trước khi chạy.
#
# Mỗi config × 3 seeds = ~36 phút (12 phút/run × 3).
# 2 forget% × 3 seeds = ~72 phút tổng.

BEST_6PER = 'C_combo_moderate'    # ⚠️ SỬA theo Cell 7 output
BEST_10PER = 'D_combo_aggressive' # ⚠️ SỬA theo Cell 7 output

# Multi-seed cho 6%
if BEST_6PER in SWEEP_CONFIGS:
    print(f"🚀 Multi-seed cho {BEST_6PER} @ 6%")
    run_sweep(forget_pct=6, configs={BEST_6PER: SWEEP_CONFIGS[BEST_6PER]}, seeds=(42, 123, 7))

# Multi-seed cho 10%
if BEST_10PER in SWEEP_CONFIGS:
    print(f"\n🚀 Multi-seed cho {BEST_10PER} @ 10%")
    run_sweep(forget_pct=10, configs={BEST_10PER: SWEEP_CONFIGS[BEST_10PER]}, seeds=(42, 123, 7))

In [ ]:
# ====================================
# CELL 9: Push results lên GitHub (Colab Secrets / Drive .git-secrets.json)
# ====================================
# Cùng logic Cell 6 của run.ipynb.

import os, json, getpass
from pathlib import Path

GITHUB_REPO = "nhnhu146/Forget-MI-LoKU"
BRANCH = "master"

def load_secrets():
    drive_path = Path("/content/drive/MyDrive/Forget-MI-Project/.git-secrets.json")
    if drive_path.exists():
        s = json.loads(drive_path.read_text())
        print(f"🔑 Credentials từ {drive_path}")
        return s.get('GITHUB_TOKEN'), s.get('GIT_EMAIL'), s.get('GIT_NAME')
    try:
        from google.colab import userdata
        t = userdata.get('GITHUB_TOKEN')
        if t:
            print("🔑 Credentials từ Colab Secrets")
            return t, userdata.get('GIT_EMAIL'), userdata.get('GIT_NAME')
    except Exception: pass
    if os.environ.get('GITHUB_TOKEN'):
        return os.environ['GITHUB_TOKEN'], os.environ.get('GIT_EMAIL', ''), os.environ.get('GIT_NAME', '')
    t = getpass.getpass("GitHub token: ").strip()
    e = input("Git email: ").strip()
    n = input("Git name: ").strip()
    return t, e, n

TOKEN, EMAIL, NAME = load_secrets()
if TOKEN and EMAIL and NAME:
    !git config user.email "{EMAIL}"
    !git config user.name "{NAME}"
    !git remote set-url origin https://{TOKEN}@github.com/{GITHUB_REPO}.git
    !git pull --rebase origin {BRANCH} 2>&1 | tail -5
    !git add experiments/ 2>/dev/null
    changes = !git diff --cached --name-only
    if changes and any(c.strip() for c in changes):
        print("\n📦 Files commit:")
        for f in changes:
            if f.strip(): print(f"   - {f}")
        !git commit -m "sweep results: 7-config hyperparameter sweep @ 6%/10%"
        !git push origin {BRANCH}
        print(f"\n✅ Pushed")
    else:
        print("ℹ️  Không có file mới.")
else:
    print("⚠️  Thiếu credentials.")